# Demo — Reading Quality Results & Choosing a Strategy

Quality results aren't a pass/fail score — each failure needs a *decision*: fix at the
**source**, fix **downstream** in the transform, or **quarantine** the bad records. This
walkthrough connects to your Cloud Lab, runs the quality suite against live Athena, rolls up
results by dimension alongside the platform's contract SLOs, and demonstrates how to classify
one failure in code; the exercise asks you to triage all failing expectations and write a
prioritized remediation plan.

## Step 0: Connect to your Cloud Lab
Run the cell below to configure your AWS Cloud Lab environment and seed the AWS Glue Data Catalog (`trailhead`) if it is not yet present in your lab account.

In [1]:
import os
os.environ["CARDINAL_LF_LIVE"] = "1"
os.environ["CARDINAL_AWS_REGION"] = "us-east-1"
# If your Vocareum AWS credentials are not already in the environment, paste them below:
# os.environ["AWS_ACCESS_KEY_ID"] = "..."
# os.environ["AWS_SECRET_ACCESS_KEY"] = "..."
# os.environ["AWS_SESSION_TOKEN"] = "..."

import boto3
import setup_aws
# Seed the Glue Data Catalog and S3 tables if not yet present in the lab account
glue = boto3.client("glue", region_name=os.environ["CARDINAL_AWS_REGION"])
try:
    glue.get_database(Name="trailhead")
except glue.exceptions.EntityNotFoundException:
    setup_aws.main()

created Glue database 'trailhead'
registered s3://123456789012-trailhead-data with Lake Formation
  customer               rows=  310  table-gaps=[]  untagged-cols=['email']
  orders                 rows= 1200  table-gaps=['owner']  untagged-cols=[]
  order_returns          rows=  106  table-gaps=[]  untagged-cols=[]
  supplier               rows=   40  table-gaps=['retention']  untagged-cols=[]
  product                rows=  200  table-gaps=['retention']  untagged-cols=['unit_cost']
  shipment               rows= 1109  table-gaps=['classification']  untagged-cols=[]
  campaign               rows=   20  table-gaps=['owner']  untagged-cols=[]
  campaign_membership    rows=  800  table-gaps=['owner', 'retention']  untagged-cols=['cust_ref']

Done. Glue database 'trailhead' seeded in s3://123456789012-trailhead-data with Lake Formation enforcement enabled.


In [2]:
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from governance_toolkit import make_catalog, lf_backend

gc = make_catalog()
print(f"Backend: {lf_backend()} -> {type(gc).__name__} (Glue DB: {gc.database}, Region: {gc.region})")

q = gc.run_quality_suite()
print("Failing expectations:", int((~q["success"]).sum()), "of", len(q))
q

Backend: live -> AwsGovernedCatalog (Glue DB: trailhead, Region: us-east-1)
Failing expectations: 8 of 9


,table,column,expectation_type,dimension,success,element_count,unexpected_count,unexpected_percent,min_value,max_value,threshold_days,unresolvable_count
0,product,category,expect_column_values_to_not_be_null,completeness,False,200,19,9.50,NaN,NaN,NaN,NaN
1,product,sustainability_class,expect_column_values_to_not_be_null,completeness,False,200,99,49.50,NaN,NaN,NaN,NaN
2,customer,email,expect_column_values_to_match_regex,validity,False,310,18,5.81,NaN,NaN,NaN,NaN
3,customer,phone,expect_column_values_to_match_regex,validity,False,310,18,5.81,NaN,NaN,NaN,NaN
4,orders,amount,expect_column_values_to_be_between,accuracy,False,1200,17,1.42,0.0,NaN,NaN,NaN
5,shipment,carbon_kg,expect_column_values_to_be_between,accuracy,False,1109,43,3.88,NaN,100.0,NaN,NaN
6,customer,customer_id,expect_column_values_to_be_unique,uniqueness,True,310,0,0.00,NaN,NaN,NaN,NaN
7,customer,created_ts,expect_recency_within_days,timeliness,False,310,12,3.87,NaN,NaN,1095.0,NaN
8,campaign_membership,mkt_consent,expect_consent_to_match_system_of_record,consistency,False,758,157,20.71,NaN,NaN,NaN,42.0


## Roll up by dimension & inspect target contract SLOs
Group by `dimension` to see where the platform is weakest across the six quality dimensions, and inspect `gc.contract_slos()` to see the target SLO for each check:

In [3]:
display(q.groupby("dimension").agg(checks=("success", "size"),
                                   failing=("success", lambda s: int((~s).sum())),
                                   worst_pct=("unexpected_percent", "max")).reset_index())
gc.contract_slos()

,dimension,checks,failing,worst_pct
0,accuracy,2,2,3.88
1,completeness,2,2,49.50
2,consistency,1,1,20.71
3,timeliness,1,1,3.87
4,uniqueness,1,0,0.00
5,validity,2,2,5.81


,table,dimension,column,slo_target,owner
0,product,completeness,category,>= 99.0% non-null,inventory-fulfillment@trailhead.example
1,product,completeness,sustainability_class,>= 99.0% non-null,inventory-fulfillment@trailhead.example
2,customer,validity,email,>= 99.5% valid format,customer-identity@trailhead.example
3,customer,validity,phone,>= 99.5% E.164 format,customer-identity@trailhead.example
4,orders,accuracy,amount,>= 99.9% non-negative,orders-team@trailhead.example
5,shipment,accuracy,carbon_kg,100% in kg (0 < kg <= 100),inventory-fulfillment@trailhead.example
6,customer,uniqueness,customer_id,100% unique,customer-identity@trailhead.example
7,customer,timeliness,created_ts,>= 98.0% within 1095d,customer-identity@trailhead.example
8,campaign_membership,consistency,mkt_consent,>= 99.0% match loyalty SoR,marketing@trailhead.example


## Inspect one failing expectation and assign a strategy
Filter to a single failure — `customer.email` on the **validity** dimension — and map it to one
of the three remediation paths (`"source"` | `"downstream"` | `"quarantine"`):

In [4]:
email_fail = q[(q["table"] == "customer") & (q["column"] == "email")].copy()
sample_remediation = {"customer.email": "quarantine"}
email_fail["key"] = email_fail["table"] + "." + email_fail["column"]
email_fail["strategy"] = email_fail["key"].map(sample_remediation)
email_fail[["key", "dimension", "expectation_type", "unexpected_count", "unexpected_percent", "strategy"]]

,key,dimension,expectation_type,unexpected_count,unexpected_percent,strategy
2,customer.email,validity,expect_column_values_to_match_regex,18,5.81,quarantine


**Why `quarantine` for `customer.email`?** Malformed addresses on a PII column create active
regulatory and operational harm — you cannot send transactional/marketing messages to `"n/a"`,
and returning corrupt contact records breaks GDPR subject-access and erasure workflows.
Quarantining isolates the bad rows immediately while the upstream producer fixes signup validation
to meet the `>= 99.5% valid format` contract SLO.

**Takeaway / your turn:** in the exercise, classify every failing `table.column` expectation as
`source`, `downstream`, or `quarantine`, and write a remediation plan prioritized by regulatory
severity and tied to `gc.contract_slos()`.